> **Note for teachers:** the `assert` lines are CI self-checks that students can ignore.
> This is one worked reference, not the answer: a student's own knob, test result and submitted file are graded on
> their evidence and their reasoning (see the rubric in `teacher-notes.md`).
> Every race here runs with the timing off except Milestone 4's `check_submission`.
> Races the solution cannot afford (175 of them) are **given data**: the probe record of plan `roar-014`, each table
> named as such; every live race (seed 0 of every stage, about 100 s of computer time) is checked against that record
> to one tick.
> The solution never races cars 72 to 143, cars k ≥ 1000, `test-circuit-seed2027` or `test-circuit-seed2028`.

# Capstone reference: `agents/unit-13.py`, and the choice that failed its test

The incumbent is `agents/unit-13.py`.
The worked choice, `TARGET_SPEED` 50.0, passed training and validation and failed its pre-registered test (23 of 24
fresh cars clean), so by Unit 13's rule the submitted file is the incumbent.
The ablation then builds `agents/unit-13.py` up from the starter, one unit's idea per stage, and the report says
what every stage bought in time and what it bought that time does not show.

## Setup

The brief's setup cell, unchanged except the scratch folder's name.

In [ ]:
import math
import sys
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from racekit import write_variant
from racekit.cars import carla_like
from racekit.check_submission import check_submission
from racekit.measure import noise_aware_better
from racekit.runner import load_submission, run_race, shim_installed
from racekit.tracks import load_track

AGENTS = Path("../racekit/agents")
STARTER, UNIT_07, UNIT_13 = AGENTS / "starter.py", AGENTS / "unit-07.py", AGENTS / "unit-13.py"

# The three files of this project.
INCUMBENT = UNIT_13  # the file you start from (or your own race-day file from Unit 13, with the book's mode switches)
CANDIDATE = None  # Milestone 2: the incumbent with your one choice, frozen before its test
SUBMITTED = None  # Milestone 2: the candidate if it passes its test, otherwise the incumbent

workdir = Path(mkdtemp(prefix="roar-capstone-sol-"))
TRACK = "monza-like"
TICK = 0.05 + 1e-9  # s: one control tick (race times are multiples of it)
DT = 0.05  # s: one control tick
CLEAN_ERROR = 2.0  # m: the clean rule's limit on the error from the race's own line
CAP = 900.0  # s: a race is cut here unless a cap is given
NO_TIMING = {"MPC_BUDGET_MS": None}  # every race here: no wall clock inside the race
UNTIMED_FILES = (STARTER, UNIT_07)  # written before Unit 10: no timing inside them to switch off
SEEDS = (0, 1, 2, 3, 4)  # the nominal car's race seeds
TRAINING_CARS = (16, 18, 8)  # Unit 13's objective: the three lowest-grip training cars
VALIDATION_CARS = range(24, 48)
TEST_CARS = range(120, 144)  # YOUR test cars: raced once, by Milestone 2's test cell only
# Cars 72-95 (Unit 13's test cars) and 96-119 (plan roar-014's) are spent; cars k >= 1000 are sealed.

track = load_track(TRACK)
with shim_installed():
    Incumbent, incumbent_name = load_submission(INCUMBENT)
incumbent = sys.modules[incumbent_name]  # the incumbent's module constants
LINE = np.array(Incumbent(track.waypoints, None).offsets)  # m: the incumbent's racing line, as offsets
CENTRELINE = np.zeros(len(track))  # m: the line of a file that races the centreline
written = []  # every variant file written so far
tested = []  # the test cars raced in this session


def variant(source, **constants):
    # Unit 1's idiom: a copy of a submission with some module constants changed, each written as its repr.
    path = workdir / f"v{len(written):03d}.py"
    written.append(path)
    return write_variant(source, path, {key: repr(value) for key, value in constants.items()})


def race(source=INCUMBENT, seed=0, k=None, laps=3, cap=CAP, test=False, **constants):
    # One race on monza-like of `source` with `constants` changed and the timing off, on the carla-like car with
    # its noise (car k of Unit 11's perturbed family if k is given, at race seed k % 5).
    assert not test or (k is not None and k in TEST_CARS), "test=True is only for your test cars 120-143"
    if k is not None:
        assert k < 72 or (test and k in TEST_CARS), "cars 72-119 are spent and k >= 1000 sealed; 120-143 are tests"
        assert k not in tested, f"car {k} has had its one test race"
        if test:
            tested.append(k)
        seed = k % 5
    if source in UNTIMED_FILES:
        path = variant(source, **constants)
    else:
        path = variant(source, **constants, **NO_TIMING)
    return run_race(path, TRACK, car=carla_like(noise=True, perturb=k), seed=seed, laps=laps, max_seconds=cap)


def error_trace(result, offsets=LINE):
    # Unit 9: the time and the cross-track error from the race's own line over the driving rows (held == 0).
    telemetry = result.telemetry
    driving = telemetry["held"] == 0
    target = offsets[(telemetry["progress"][driving] + 1) % len(offsets)]
    return telemetry["t"][driving], telemetry["offset"][driving] - target


def line_error(result, offsets=LINE):
    return float(np.max(np.abs(error_trace(result, offsets)[1])))


def clean(result, offsets=LINE):
    return result.finished and result.collisions == 0 and line_error(result, offsets) <= CLEAN_ERROR


def in_ticks(times):
    # Unit 12's habit: race times (and sums of them) as whole ticks before any comparison.
    return [round(t / DT) for t in times]


print(f"incumbent {INCUMBENT.name}: TARGET_SPEED {incumbent.TARGET_SPEED}, STEERING {incumbent.STEERING!r}, "
      f"{len(LINE)} line offsets")

## Milestone 1

### The bar

The shipped starter is raced once, capped at 120 s.
The `TARGET_SPEED` grid is the probe record of plan `roar-014` (given data); only the bar's own seed-0 race is
raced live, and seeds 1 to 4 of the bar are the record's (the routed premise C14.1 re-races all five).

**Prediction.** On the grip car the starter's steering at 20 m/s should crash before 120 s.
Slower settings should be clean; the fastest clean grid speed should be the bar, about 1,200 s.

In [ ]:
# 1. The shipped starter, cut at 120 s.
shipped_starter = race(STARTER, cap=120.0)
print(f"starter as shipped (TARGET_SPEED {20.0}): {shipped_starter.total_time:.2f} s, finished "
      f"{shipped_starter.finished}, {shipped_starter.collisions} collision(s), largest centreline error "
      f"{line_error(shipped_starter, CENTRELINE):.3f} m, clean {clean(shipped_starter, CENTRELINE)}")
why_not_clean = (
    "Unit 1 raced this file on KinematicCar, a car whose wheels go exactly where they point, so its proportional "
    "steering at 20 m/s held monza-like's corners; carla_like(noise=True) is a dynamic car with tyres that slip, a "
    "one-tick actuator lag and noise, the car Unit 6's hook spun with the same file. "
    "The starter steers on the heading to the next waypoint with no look-ahead and no corner speed, so at 20 m/s "
    "every correction on the grip car arrives late and grows, and it crash-loops in the first few hundred waypoints: "
    "the file did not change, the car did."
)
print(why_not_clean)

# 3. The TARGET_SPEED grid at race seed 0, cut at 2,400 s. GIVEN DATA: the probe record of plan roar-014 (p3),
# speed -> (the seed-0 time in s, collisions over seeds 0-4, the largest centreline error over seeds 0-4 in m,
# clean on all of seeds 0-4). Only the bar's own seed-0 race (14 m/s) is raced live below.
GRID_RECORD = {
    10.0: (1650.50, 0, 0.150, True),
    12.0: (1390.45, 0, 0.309, True),
    14.0: (1203.45, 0, 0.706, True),
    16.0: (1068.65, 0, 2.695, False),  # finishes with 0 collisions but 2.695 m from the centreline
}
# The bar at seeds 0-4 (p3; C14.1 re-raced the same five in the routed premise): the five times in s.
BAR_RECORD = [1203.45, 1203.55, 1203.90, 1203.85, 1203.20]
BAR_RECORD_CLEAN = 5  # clean of 5, 0 collisions, the largest centreline error 0.706 m

grid_table = [(speed, time, collisions, error, is_clean)
              for speed, (time, collisions, error, is_clean) in GRID_RECORD.items()]
print(" speed   time (s)  collisions  error (m)  clean")
for speed, time, collisions, error, is_clean in grid_table:
    print(f"{speed:6.1f}  {time:9.2f}  {collisions:10d}  {error:9.3f}  {is_clean}")
bar_speed = max(speed for speed, time, collisions, error, is_clean in grid_table if is_clean)

# 4. The bar: the starter at bar_speed. Seed 0 is raced live (about 6 s of computer time); seeds 1-4 are the record's.
grid = {bar_speed: race(STARTER, cap=2400.0, TARGET_SPEED=bar_speed)}
bar_seed0 = grid[bar_speed]
bar_results = [bar_seed0]  # the live race; the other four seeds are given data
bar_times = BAR_RECORD
bar_clean = BAR_RECORD_CLEAN
print(f"the bar, TARGET_SPEED {bar_speed}, seed 0 live: {bar_seed0.total_time:.2f} s, {bar_seed0.collisions} "
      f"collisions, centreline error {line_error(bar_seed0, CENTRELINE):.3f} m, clean {clean(bar_seed0, CENTRELINE)}")
print(f"the bar on seeds 0-4 (record): {bar_times}, median {np.median(bar_times):.2f} s, "
      f"spread {max(bar_times) - min(bar_times):.2f} s, clean {bar_clean}/5")

assert not shipped_starter.finished and shipped_starter.collisions >= 1  # C14.0
assert bar_speed == 14.0
assert abs(bar_seed0.total_time - BAR_RECORD[0]) <= TICK and bar_seed0.collisions == 0  # 1203.45 s
assert clean(bar_seed0, CENTRELINE) and abs(line_error(bar_seed0, CENTRELINE) - 0.69) <= 0.01

## Milestone 2

### Your choice, and its test

The worked choice is the record's: `TARGET_SPEED` from `agents/unit-13.py`, by Unit 13's protocol, with its
validation and its pre-registered test given as data.
The walk is replayed here on whole ticks; `objective()` is the function that raced it (five races per value), shown
and not called.
The test cars of this worked choice were 96 to 119, raced once by probe `p10`: the solution never races them, nor a
student's own cars 120 to 143.

**Prediction.** `TARGET_SPEED` was set in Unit 4 and never re-tuned after the grip, the line and MPC raised the
corner speeds, so a few steps up should beat the shipped 45 until one is not clean; one step inside that edge should be
clean on the validation cars.
The test may still fail: a margin thins a risk, it does not remove it.

In [ ]:
knob = "TARGET_SPEED"
knob_reason = (
    "TARGET_SPEED is Unit 4's top speed, set to 45 m/s on the kinematic car and never re-tuned after: since then "
    "the measured grip (Unit 8), the racing line (Unit 9) and the MPC (Unit 10) all raised what the car can carry "
    "into a corner, and Unit 13's re-tune moved only MU_FRACTION, A_DEC and A_ACC. "
    "On monza-like's long straights the profile sits at TARGET_SPEED, so if the car can now go faster there, this is "
    "where the time is."
)
grid_step = 2.5  # m/s, written down before any race

# GIVEN DATA: the probe record of plan roar-014 (p8a, p8b): value -> ((the two per-replicate sums in s, exactly as
# computed), {race that was not clean: (collisions, line error in m)}). Nominal seeds 0 and 1 plus cars 16, 18, 8.
P8 = {
    45.0: ((1877.8000000000002, 1877.8000000000002), {}),  # the shipped value
    47.5: ((1831.2, 1831.25), {}),
    50.0: ((1795.5500000000002, 1795.65), {}),
    52.5: ((1767.3000000000002, 1767.25), {}),
    55.0: ((1816.25, 1816.0500000000002), {"car 8": (1, 5.273)}),  # car 8 collides: 508.50 s
}
P8_NOMINAL = {45.0: (468.55, 468.55), 47.5: (456.85, 456.90), 50.0: (447.80, 447.90), 52.5: (440.75, 440.70),
              55.0: (434.75, 434.55)}  # s: the nominal races at seeds 0 and 1
# The second knob, the same objective (p8a): PROFILE_LEAD_S (shipped 0.3) and MPC_Q_E (shipped 0.3).
SECOND_KNOB = {
    "PROFILE_LEAD_S": {0.3: P8[45.0], 0.2: ((1876.9, 1877.0), {}), 0.4: ((1881.3000000000002, 1881.25), {}),
                       0.1: ((1874.95, 1875.0500000000002), {}), 0.0: ((1872.5500000000002, 1872.65), {})},  # p11, p11b
    "MPC_Q_E": {0.3: P8[45.0], 0.2: ((1878.0500000000002, 1878.15), {}),
                0.5: ((2159.15, 2159.1), {"car 16": (1, 2.169), "car 8": (1, 1.632)})},
}
# GIVEN DATA (p11c): PROFILE_LEAD_S 0.0 on the validation cars 24-47 (race seed k % 5): the 24 times in s, and the
# cars not clean: (collisions, time in s, line error in m).
LEAD_VALIDATION_TIMES = [707.25, 465.95, 467.6, 467.7, 466.95, 469.6, 468.25, 469.15, 469.5, 466.2, 467.35, 468.9,
                         467.0, 466.55, 469.7, 468.25, 468.45, 466.45, 467.35, 467.4, 467.5, 465.95, 466.55, 468.8]
LEAD_VALIDATION_NOT_CLEAN = {24: (2, 707.25, 3.975), 35: (0, 468.90, 2.009)}


def objective(**constants):
    # The training objective, as the record raced it: five races, two per-replicate sums and the races not clean.
    sums = []
    failures = {}
    nominal = {r: race(seed=r, **constants) for r in (0, 1)}
    cars = {k: race(k=k, **constants) for k in TRAINING_CARS}
    for r in (0, 1):
        sums.append(nominal[r].total_time + sum(cars[k].total_time for k in TRAINING_CARS))
    for name, result in [(f"seed {r}", nominal[r]) for r in (0, 1)] + [(f"car {k}", cars[k]) for k in TRAINING_CARS]:
        if not clean(result):
            failures[name] = (result.collisions, line_error(result))
    return sums, failures


def beats(a, b):
    # Unit 13's rule: Unit 1's rule on whole ticks (Unit 12's floating-point trap).
    return noise_aware_better(in_ticks(a), in_ticks(b))


# The walk (this replays the record; objective() above is what raced it, and is not called here).
training = P8
start = 45.0
accepted, edge = [], None
last = start
value = start + grid_step
while value in training:
    sums, failures = training[value]
    if failures:
        edge = max(v for v in training if v < value and not training[v][1])  # the highest clean value
        break
    if not beats(sums, training[last][0]):
        break
    accepted.append(value)
    last = value
    value = value + grid_step
margin_value = edge - grid_step if edge is not None else last
print(" value  sums (ticks)     nominal s0, s1 (s)  clean  beats the value before")
values = list(training)
for i, value in enumerate(values):
    sums, failures = training[value]
    verdict = "-" if i == 0 or failures else beats(sums, training[values[i - 1]][0])
    print(f"{value:6.1f}  {in_ticks(sums)!s:16s}  {P8_NOMINAL[value]!s:18s}  {not failures!s:5s}  {verdict}"
          f"  {failures if failures else ''}")
print(f"accepted {accepted}; the edge {edge}; Unit 11's margin: {margin_value}")
for name, record in SECOND_KNOB.items():
    shipped = record[0.3][0]
    for value, (sums, failures) in record.items():
        if value != 0.3:
            gain = (sum(shipped) - sum(sums)) / 2
            print(f"second knob {name} {value}: clean {not failures}, beats the shipped 0.3 "
                  f"{not failures and beats(sums, shipped)}, mean gain per sum {gain:.2f} s",
                  failures if failures else "")
# PROFILE_LEAD_S walked down (0.4 is slower): 0.2, 0.1 and 0.0 each beat the value before and all are clean. A lead
# below 0 s is the same lead (the agent floors it at LEAD_MIN_M), so 0.0 ends the walk with no value that is not
# clean: no edge, no margin, keep the last value that beat (the brief's rule), and validate it.
lead = SECOND_KNOB["PROFILE_LEAD_S"]
lead_walk = [0.3, 0.2, 0.1, 0.0]
lead_beats = [beats(lead[b][0], lead[a][0]) for a, b in zip(lead_walk, lead_walk[1:])]
lead_value = lead_walk[-1]
lead_clean = len(LEAD_VALIDATION_TIMES) - len(LEAD_VALIDATION_NOT_CLEAN)
print(f"PROFILE_LEAD_S walk {lead_walk}: each beats the last {lead_beats}; kept {lead_value}; on the validation "
      f"cars clean {lead_clean}/24, median {np.median(LEAD_VALIDATION_TIMES):.2f} s, not clean {LEAD_VALIDATION_NOT_CLEAN}")
second_knob_evidence = (
    "PROFILE_LEAD_S (Unit 7's lead, shipped 0.3 s): 0.2, 0.1 and 0.0 each beat the value before on the training "
    "objective (5 s per sum in all, about 1.3 s a race) and all were clean, so with no value that is not clean there "
    "was no edge and no margin: 0.0 was kept and raced on the validation cars, where it was clean on 22 of 24 (car 24 "
    "collided twice; car 35 ended 2.009 m from its line). So no second candidate: the shipped 0.3 stays, and the "
    "evidence stops at validation, as it must (the test cars test one file only). "
    "MPC_Q_E (Unit 10's error weight, shipped 0.3): 0.2 is slower and 0.5 is not clean (cars 16 and 8 collide), so "
    "the shipped 0.3 stays."
)
print(second_knob_evidence)

assert accepted == [47.5, 50.0, 52.5] and edge == 52.5 and margin_value == 50.0
assert [in_ticks(training[v][0]) for v in (45.0, 50.0)] == [[37556, 37556], [35911, 35913]]
assert beats(SECOND_KNOB["PROFILE_LEAD_S"][0.2][0], P8[45.0][0])
assert not beats(SECOND_KNOB["PROFILE_LEAD_S"][0.4][0], P8[45.0][0])
assert not beats(SECOND_KNOB["MPC_Q_E"][0.2][0], P8[45.0][0]) and SECOND_KNOB["MPC_Q_E"][0.5][1]
assert lead_beats == [True, True, True] and lead_clean == 22 and sorted(LEAD_VALIDATION_NOT_CLEAN) == [24, 35]

In [ ]:
# 2. Validation. GIVEN DATA: the probe record of plan roar-014 (p8v): the 24 validation cars k = 24-47 (race seed
# k % 5, 3 laps), value -> {k: time in s}, and {k: (collisions, line error in m)} for each car that was not clean.
VALIDATION_TIMES = {
    50.0: [447.60, 445.85, 448.30, 448.55, 447.50, 450.75, 449.25, 450.40, 450.60, 446.25, 447.95, 449.85,
           447.40, 446.90, 451.00, 449.05, 449.40, 446.40, 448.10, 448.05, 448.15, 446.10, 446.55, 449.85],
    52.5: [818.60, 438.55, 441.15, 441.35, 440.20, 443.95, 442.20, 443.40, 443.60, 438.85, 440.80, 443.00,
           440.05, 439.45, 444.10, 442.05, 442.40, 439.05, 440.80, 808.50, 440.95, 438.85, 795.85, 442.70],
}
VALIDATION_NOT_CLEAN = {50.0: {}, 52.5: {24: (2, 3.298), 43: (1, 2.001), 46: (1, 2.547)}}
VALIDATION_MAX_ERROR = {50.0: 1.341, 52.5: 3.298}  # m: over the 24 cars

validation = {k: time for k, time in zip(VALIDATION_CARS, VALIDATION_TIMES[margin_value])}
validation_clean = len(VALIDATION_CARS) - len(VALIDATION_NOT_CLEAN[margin_value])
has_candidate = validation_clean == len(VALIDATION_CARS)
for value in (margin_value, edge):
    value_clean = len(VALIDATION_CARS) - len(VALIDATION_NOT_CLEAN[value])
    print(f"TARGET_SPEED {value} on the validation cars: clean {value_clean}/24, median "
          f"{np.median(VALIDATION_TIMES[value]):.2f} s, largest line error {VALIDATION_MAX_ERROR[value]} m, "
          f"not clean {VALIDATION_NOT_CLEAN[value]}")
print("has a candidate:", has_candidate)

# 3. Freeze. The solution writes its candidate into its own scratch folder (a student writes it to a folder of their
# own outside the public repository, and edits its header): agents/unit-13.py with TARGET_SPEED 50.0, nothing else.
CANDIDATE = write_variant(INCUMBENT, workdir / "candidate.py", {knob: repr(margin_value)})
print("candidate:", CANDIDATE)

# 4. The pre-registration, as the record made it (plan roar-014 v2, commit dc170c9, before any race of the test).
PRE_REGISTRATION = {
    "file": "agents/capstone.py: agents/unit-13.py with TARGET_SPEED = 50.0 and nothing else changed",
    "change": "TARGET_SPEED 45.0 -> 50.0 (one grid step inside the edge 52.5)",
    "cars": "96-119, race seed k % 5, 3 laps, monza-like, carla_like(noise=True), timing off, raced once (p10)",
    "rule": "not written in dc170c9 (its mistake); the rule that applies is Unit 13's, merged in 44f9753 (#168) "
            "before dc170c9: a file is submitted only if it is clean on every car it was never tuned on (N/N)",
    "decision": "dc170c9: 'the result is reported as it comes: no choice in this plan may change after it'",
    "written": "2026-10-29 22:25:06 -0700 (dc170c9), before p10 ran",
}
for key, value in PRE_REGISTRATION.items():
    print(f"{key:>9}: {value}")

In [ ]:
# The test. GIVEN DATA, never raced here: the probe record of plan roar-014 (p10), the pre-registered candidate on
# cars 96-119, raced once: k -> (time in s, collisions, the largest line error in m). Every race finished.
TEST_RECORD = {
    96: (448.35, 0, 1.266), 97: (449.65, 0, 1.286), 98: (445.35, 0, 1.224), 99: (446.80, 0, 1.201),
    100: (445.30, 0, 1.193), 101: (448.65, 0, 1.227), 102: (450.80, 0, 1.216), 103: (450.15, 0, 1.228),
    104: (448.35, 0, 1.186), 105: (445.55, 0, 1.161), 106: (448.65, 0, 1.210), 107: (449.10, 0, 1.323),
    108: (449.10, 0, 1.235), 109: (448.25, 0, 1.245), 110: (448.30, 0, 1.281), 111: (445.85, 0, 1.255),
    112: (450.00, 0, 1.248), 113: (451.15, 0, 1.168), 114: (450.90, 0, 1.157), 115: (448.45, 0, 1.106),
    116: (446.50, 0, 1.191), 117: (590.05, 2, 4.485), 118: (448.50, 0, 1.120), 119: (448.45, 0, 1.259),
}
test_results = TEST_RECORD  # the solution's test cars were 96-119; it never races them, nor yours (120-143)
test_failures = {k: (collisions, time, error) for k, (time, collisions, error) in TEST_RECORD.items()
                 if collisions > 0 or error > CLEAN_ERROR}
test_clean = len(TEST_RECORD) - len(test_failures)
passed = test_clean == len(TEST_RECORD)  # the rule: clean on every one of the 24
SUBMITTED = CANDIDATE if passed else INCUMBENT
SUBMITTED_STAGE = "S10" if passed else "S9"
clean_times = [time for k, (time, collisions, error) in TEST_RECORD.items() if k not in test_failures]
print(f"test cars 96-119: clean {test_clean}/24; not clean (collisions, time, line error): {test_failures}")
print(f"the 23 clean races: {min(clean_times):.2f}-{max(clean_times):.2f} s; median of all 24 "
      f"{np.median([time for time, collisions, error in TEST_RECORD.values()]):.2f} s")
print(f"the rule (N/N) passed: {passed}; submitted: {SUBMITTED} ({SUBMITTED_STAGE})")
print("the incumbent's own test: Unit 13's cars 72-95, clean 24/24 (median 468.65 s)")

assert has_candidate and validation_clean == 24 and len(VALIDATION_CARS) - len(VALIDATION_NOT_CLEAN[52.5]) == 21
assert test_clean == 23 and list(test_failures) == [117] and not passed
assert SUBMITTED == INCUMBENT == UNIT_13 and SUBMITTED_STAGE == "S9"

## Milestone 3

### Build it up: what every stage bought

The submitted file is `agents/unit-13.py`, so the ablation is the reference grid S1 to S9, with the candidate as
row S10, marked **not submitted**.
Seed 0 of every stage is raced live (three cells, about 80 s of computer time) and checked against the record to one
tick, its clean flag and its line error to 0.01 m; the five-seed lists are the record's (given data).
The leave-one-out is the record's too.

**Prediction.** Most of the time should be bought early, by the stages that let the car go faster (the cap and the
profile), and less and less after.
MPC and the fallback should buy little or no time on the nominal car: what they buy is margin.

In [ ]:
# The reference grid of plan roar-014: each stage's constants, written over the incumbent (agents/unit-13.py).
OFF = {"RESPAWN_IMPULSE": 1e300}  # the respawn fallback off: no impulse is ever this large
PRE_TUNE = {"MU_FRACTION": 0.54, "A_DEC": 5.0, "A_ACC": 4.0}  # the knobs before Unit 13's re-tune
NO_LINE_NO_GRIP = {"MU_G": None, "LINE_CONTROL": None}
REFERENCE_GRID = {
    "S1": {**OFF, **PRE_TUNE, **NO_LINE_NO_GRIP, "STEERING": "pid", "SPEED_MODE": "flat", "TARGET_SPEED": 14.0},
    "S2a": {**OFF, **PRE_TUNE, **NO_LINE_NO_GRIP, "STEERING": "pursuit", "SPEED_MODE": "flat", "TARGET_SPEED": 14.0},
    "S2b": {**OFF, **PRE_TUNE, **NO_LINE_NO_GRIP, "STEERING": "pursuit", "SPEED_MODE": "flat", "TARGET_SPEED": 15.0},
    "S3": {**OFF, **PRE_TUNE, **NO_LINE_NO_GRIP, "STEERING": "pursuit", "SPEED_MODE": "cap", "A_LAT": 4.6,
           "TARGET_SPEED": 32.5},
    "S4": {**OFF, **PRE_TUNE, **NO_LINE_NO_GRIP, "STEERING": "pursuit", "SPEED_MODE": "profile", "A_LAT": 4.6,
           "TARGET_SPEED": 45.0},
    "S5": {**OFF, **PRE_TUNE, "LINE_CONTROL": None, "STEERING": "pursuit", "SPEED_MODE": "profile", "A_LAT": 4.6,
           "TARGET_SPEED": 45.0},
    "S6": {**OFF, **PRE_TUNE, "STEERING": "pursuit", "SPEED_MODE": "profile", "A_LAT": 4.6, "TARGET_SPEED": 45.0},
    "S7": {**OFF, **PRE_TUNE, "STEERING": "mpc", "SPEED_MODE": "profile", "A_LAT": 4.6, "TARGET_SPEED": 45.0},
    "S8": {**PRE_TUNE, "STEERING": "mpc", "SPEED_MODE": "profile", "A_LAT": 4.6, "TARGET_SPEED": 45.0},
    "S9": {},  # the incumbent as shipped: Unit 13's re-tune
}
STAGE_IDEA = {
    "S0": "the bar: starter.py at its fastest clean TARGET_SPEED",
    "S1": "Unit 4: PID steering, a flat 14 m/s, the centreline, no measured grip",
    "S2a": "Unit 5: pure pursuit, the same 14 m/s",
    "S2b": "Unit 5's speed: pursuit at 15 m/s",
    "S3": "Unit 6: the corner cap (A_LAT 4.6, TARGET_SPEED 32.5)",
    "S4": "Unit 7: the speed profile (TARGET_SPEED 45)",
    "S5": "Unit 8: the measured grip (MU_G, MU_FRACTION 0.54)",
    "S6": "Unit 9: the racing line",
    "S7": "Unit 10: MPC steering",
    "S8": "Unit 11: the respawn fallback on",
    "S9": "Unit 13: the race-day re-tune (the incumbent)",
    "S10": "your choice (the candidate)",
}
STAGE_CAP = {"S1": 2400.0, "S2a": 1800.0, "S2b": 1800.0, "S3": 1800.0}  # s; every other stage: CAP
STAGE_ORDER = ["S0", "S1", "S2a", "S2b", "S3", "S4", "S5", "S6", "S7", "S8", "S9", "S10"]
CENTRELINE_STAGES = ("S0", "S1", "S2a", "S2b", "S3", "S4", "S5")  # no line raced: errors from the centreline

# GIVEN DATA: the probe record of plan roar-014 (p2, p3, p7, p10): every stage at race seeds 0-4 (3 laps, timing off),
# stage -> (the five times in s, in seed order; clean of 5; the largest line error of the five in m). All 0 collisions.
STAGE_RECORD = {
    "S0": ([1203.45, 1203.55, 1203.90, 1203.85, 1203.20], 5, 0.706),
    "S1": ([1219.90, 1219.75, 1220.10, 1219.70, 1219.50], 5, 0.434),
    "S2a": ([1220.05, 1219.90, 1220.20, 1219.90, 1219.65], 5, 0.429),
    "S2b": ([1138.60, 1138.50, 1138.95, 1138.85, 1138.90], 5, 1.042),
    "S3": ([617.05, 616.95, 616.95, 617.10, 617.05], 5, 0.753),
    "S4": ([514.95, 514.95, 515.05, 515.15, 514.95], 5, 1.575),
    "S5": ([510.20, 510.15, 510.25, 510.25, 510.10], 5, 1.734),
    "S6": ([482.05, 481.95, 482.05, 482.00, 482.00], 5, 1.529),
    "S7": ([482.05, 482.00, 482.05, 482.00, 482.00], 5, 0.976),
    "S8": ([482.05, 482.00, 482.05, 482.00, 482.00], 5, 0.976),
    "S9": ([468.55, 468.55, 468.55, 468.65, 468.60], 5, 1.009),
    "S10": ([447.80, 447.90, 447.95, 448.00, 447.80], 5, 1.141),
}
# The re-measured seed-0 race of every stage (plan roar-014, Phase A): (time in s, clean, the largest line error in m).
SEED0_RECORD = {
    "S1": (1219.90, True, 0.395), "S2a": (1220.05, True, 0.429), "S2b": (1138.60, True, 1.021),
    "S3": (617.05, True, 0.638), "S4": (514.95, True, 1.572), "S5": (510.20, True, 1.734),
    "S6": (482.05, True, 1.498), "S7": (482.05, True, 0.942), "S8": (482.05, True, 0.942),
    "S9": (468.55, True, 0.988), "S10": (447.80, True, 1.141),
}
seed0 = {"S0": bar_seed0}  # the live seed-0 race of every stage (S0's is Milestone 1's)


def race_stage(stage):
    # One live race at seed 0: S1-S9 from the grid over the incumbent, S10 the frozen candidate.
    if stage == "S10":
        return race(CANDIDATE, seed=0)
    return race(INCUMBENT, seed=0, cap=STAGE_CAP.get(stage, CAP), **REFERENCE_GRID[stage])


def check_stage(stage):
    # The live race against the record: the time to one tick, the clean flag, the line error to 0.01 m.
    result = seed0[stage]
    offsets = CENTRELINE if stage in CENTRELINE_STAGES else LINE
    time, is_clean, error = result.total_time, clean(result, offsets), line_error(result, offsets)
    print(f"{stage:4s} seed 0: {time:8.2f} s (record {SEED0_RECORD[stage][0]:.2f}), clean {is_clean}, "
          f"line error {error:.3f} m (record {SEED0_RECORD[stage][2]:.3f}), {result.collisions} collisions")
    assert abs(time - SEED0_RECORD[stage][0]) <= TICK and is_clean == SEED0_RECORD[stage][1]
    assert abs(error - SEED0_RECORD[stage][2]) <= 0.01 and result.collisions == 0
    assert abs(time - STAGE_RECORD[stage][0][0]) <= TICK  # and the five-seed record's seed 0

In [ ]:
# The live seed-0 races, part 1: Units 4 to 6 (about 30 s of computer time).
for stage in ("S1", "S2a", "S2b", "S3"):
    seed0[stage] = race_stage(stage)
    check_stage(stage)

In [ ]:
# The live seed-0 races, part 2: Units 7 to 10 (about 21 s).
for stage in ("S4", "S5", "S6", "S7"):
    seed0[stage] = race_stage(stage)
    check_stage(stage)

In [ ]:
# The live seed-0 races, part 3: Unit 11, the incumbent and the candidate (about 30 s).
for stage in ("S8", "S9", "S10"):
    seed0[stage] = race_stage(stage)
    check_stage(stage)
s7, s8 = seed0["S7"].telemetry, seed0["S8"].telemetry
fallback_identical = len(s7) == len(s8) and all(np.sum(s7[name] != s8[name]) == 0 for name in s7.dtype.names)
print("S7 and S8 drive the same race, row for row:", fallback_identical)
assert fallback_identical

In [ ]:
# The ablation table, from the five-seed record (seed 0 of every row was raced live above and matches it).
stage_results = {stage: STAGE_RECORD[stage][0] for stage in STAGE_ORDER}  # the five times of each stage
s10_submitted = SUBMITTED_STAGE == "S10"
ablation = []
previous = None
for stage in STAGE_ORDER:
    times, clean_count, max_error = STAGE_RECORD[stage]
    row = {"stage": stage, "idea": STAGE_IDEA[stage], "clean": clean_count, "median": float(np.median(times)),
           "spread": max(times) - min(times), "time bought": None, "verdict": None, "max error": max_error}
    if previous is not None:
        row["time bought"] = previous["median"] - row["median"]
        row["verdict"] = noise_aware_better(times, stage_results[previous["stage"]])
    ablation.append(row)
    previous = row
row_of = {row["stage"]: row for row in ablation}

print(f"{'stage':5s} {'idea':70s} clean  median   spread  bought  better  max error")
for row in ablation:
    bought = "" if row["time bought"] is None else f"{row['time bought']:7.2f}"
    verdict = "" if row["verdict"] is None else str(row["verdict"])
    mark = ""
    if row["stage"] == SUBMITTED_STAGE:
        mark = "  <- submitted"
    elif row["stage"] == "S10":
        mark = "  <- the candidate: NOT submitted (test cars 23/24)"
    error_from = "(c)" if row["stage"] in CENTRELINE_STAGES else ""
    print(f"{row['stage']:5s} {row['idea']:70s} {row['clean']}/5  {row['median']:7.2f}  {row['spread']:5.2f}  "
          f"{bought:>7s}  {verdict:6s}  {row['max error']:.3f} {error_from}{mark}")
print("(c): the error from the centreline (no line raced); seeds 0-4 on every row; 0 collisions everywhere")

positions = list(range(len(STAGE_ORDER)))
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
for position, stage in zip(positions, STAGE_ORDER):
    axes[0].scatter([position] * 5, stage_results[stage], color="C0", s=12,
                    label="the five races (seeds 0-4)" if position == 0 else None)
axes[0].plot(positions, [row["median"] for row in ablation], color="C1", marker="o", label="median")
axes[0].set_ylabel("race time (s)")
axes[0].set_title("The ablation of agents/unit-13.py on monza-like, carla_like(noise=True): time and margin by stage")
axes[0].legend()
axes[1].plot(positions, [row["max error"] for row in ablation], color="C2", marker="o",
             label="largest line error of the five (S0-S5 from the centreline)")
axes[1].axhline(CLEAN_ERROR, color="grey", linestyle="--", label="the 2 m clean limit")
axes[1].set_xlabel("stage: " + ", ".join(f"{position} {stage}" for position, stage in zip(positions, STAGE_ORDER))
                   + " (S10: the candidate, not submitted)")
axes[1].set_ylabel("line error (m)")
axes[1].legend()
plt.show()

# Leave one out, from the submitted file (S9). GIVEN DATA: the probe record of plan roar-014 (p2): removal -> (the
# five times in s, clean of 5). "no tune" puts back the knobs before Unit 13's re-tune (PRE_TUNE).
LEAVE_ONE_OUT_RECORD = {
    "no line": ([495.50, 495.35, 495.40, 495.55, 495.35], 5),  # LINE_CONTROL = None, scored on the centreline
    "no grip": ([486.25, 486.15, 486.45, 486.30, 486.20], 5),  # MU_G = None (the cap's A_LAT 4.6)
    "no MPC": ([468.55, 468.55, 468.60, 468.65, 468.60], 5),  # STEERING = "pursuit"
    "no tune": ([482.05, 482.00, 482.05, 482.00, 482.00], 5),  # PRE_TUNE
}
submitted_times = stage_results[SUBMITTED_STAGE]
submitted_median = row_of[SUBMITTED_STAGE]["median"]
leave_one_out = []
for removed, (times, clean_count) in LEAVE_ONE_OUT_RECORD.items():
    leave_one_out.append({"removed": removed, "median": float(np.median(times)),
                          "cost": float(np.median(times)) - submitted_median,
                          "verdict": noise_aware_better(submitted_times, times), "clean": clean_count})
BUILD_UP_STAGE = {"no line": "S6", "no grip": "S5", "no MPC": "S7", "no tune": "S9"}  # where each idea was added
print(f"\nleave one out from {SUBMITTED_STAGE} ({submitted_median:.2f} s):")
for row in leave_one_out:
    bought = row_of[BUILD_UP_STAGE[row["removed"]]]["time bought"]
    print(f"  {row['removed']:8s} median {row['median']:.2f} s, costs {row['cost']:6.2f} s, the full file better "
          f"{row['verdict']!s:5s}, clean {row['clean']}/5 | the same idea bought {bought:6.2f} s in the build-up")

# Why S1 costs time (Milestone 5, answer 3): the live seed-0 races of S0 and S1, distance and speed.
for stage in ("S0", "S1"):
    telemetry = seed0[stage].telemetry
    distance = float(np.sum(np.hypot(np.diff(telemetry["x"]), np.diff(telemetry["y"]))))
    print(f"{stage} seed 0: {distance / 1000:.2f} km driven, mean speed {np.mean(telemetry['v']):.3f} m/s, "
          f"largest {np.max(telemetry['v']):.2f} m/s")
mean_speed = {stage: float(np.mean(seed0[stage].telemetry["v"])) for stage in ("S0", "S1")}

expected_bought = {"S1": -16.20, "S2a": -0.15, "S2b": 81.05, "S3": 521.80, "S4": 102.10, "S5": 4.75, "S6": 28.20,
                   "S7": 0.00, "S8": 0.00, "S9": 13.45, "S10": 20.65}
expected_verdict = {"S1": False, "S2a": False, "S2b": True, "S3": True, "S4": True, "S5": True, "S6": True,
                    "S7": False, "S8": False, "S9": True, "S10": True}
for stage in STAGE_ORDER[1:]:
    assert abs(row_of[stage]["time bought"] - expected_bought[stage]) <= 1e-6, stage
    assert row_of[stage]["verdict"] == expected_verdict[stage], stage
assert all(row["clean"] == 5 for row in ablation) and not s10_submitted
assert mean_speed["S0"] > mean_speed["S1"] > 14.0
assert [round(row["cost"], 2) for row in leave_one_out] == [26.85, 17.70, 0.05, 13.45]
assert [row["verdict"] for row in leave_one_out] == [True, True, False, True]

## Milestone 4

### Qualify

The gate is computed on the five-seed lists (given data; the routed premise C14.1 raced all ten), and
`agents/unit-07.py`'s five races are the record's.
`check_submission` is the one timed race: it is run live, and its p99 is printed, never asserted.

**Prediction.** `agents/unit-13.py` should beat the bar by more than 700 s and `unit-07.py` by tens of seconds, and
pass the check on its own track.

In [ ]:
# 1. Against the bar: the submitted file's five races (its row) against the bar's five, both from the record
# (C14.1 re-raced all ten in the routed premise; their seed 0s were raced live above).
submitted_times = stage_results[SUBMITTED_STAGE]
all_clean = row_of[SUBMITTED_STAGE]["clean"] == 5 and bar_clean == 5  # every one of the ten at 0 collisions
beats_bar = all_clean and noise_aware_better(submitted_times, bar_times)
bar_margin = float(np.median(bar_times) - np.median(submitted_times))
print(f"{SUBMITTED.name} {submitted_times} against the bar {bar_times}: all ten clean {all_clean}, "
      f"noise_aware_better {noise_aware_better(submitted_times, bar_times)}, margin {bar_margin:.2f} s")

# 2. Against Part 1. GIVEN DATA: the probe record of plan roar-014 (p2): agents/unit-07.py at seeds 0-4, 3 laps
# (the centreline; no timing to switch off): the five times in s, clean of 5 against the centreline.
UNIT07_RECORD = ([515.10, 515.10, 515.20, 515.25, 515.10], 5)
unit07_times, unit07_clean = UNIT07_RECORD
beats_unit07 = noise_aware_better(submitted_times, unit07_times)
print(f"unit-07.py {unit07_times} (median {np.median(unit07_times):.2f} s, clean {unit07_clean}/5): the submitted "
      f"file beats it {beats_unit07}, by {np.median(unit07_times) - np.median(submitted_times):.2f} s")

# 3. The submission check: the file exactly as submitted, the timing ON, on its own track (about 10 s).
# Run it when nothing else is running: the p99 is this machine's, under its load, and is printed, never asserted.
report = check_submission(SUBMITTED, track="monza-like")
print(report)
print(f"step() p99 on this machine, now: {report.step_p99_ms:.3f} ms (Unit 13's budget: 5 ms, race machine, idle)")
# Asserted: the file parses, has the official shape and finishes the race. The p99 budget is the one stage that
# depends on this machine's load, so report.ok (which includes it) is printed, not asserted.
check_passed = (not report.stages["parse"] and not report.stages["shape"] and report.race is not None
                and report.race.finished)
print("check_submission ok on this machine, now (p99 included):", report.ok)

# 4. The packaging checklist (Unit 13, section 6), each with what the solution found.
checklist = [
    ("One file, agents/unit-13.py, in the official shape (class RoarCompetitionSolution, async initialize and step): "
     "check_submission's shape stage passed."),
    ("Python 3.8: check_submission parses it at 3.8 and passed; imports are only __future__, math, time, typing, "
     "numpy and roar_py_interface (Unit 13, Exercise 8)."),
    ("The values: MU_FRACTION 0.64, A_DEC 4.5, A_ACC 5.0 on their grids, written with repr, each reads back as the "
     "value raced; TARGET_SPEED stays 45.0 (the candidate's 50.0 failed its test and is not in the file)."),
    ("The header comment names the six constants Unit 13 changed and why; nothing was changed after (the submitted "
     "file is the incumbent, byte for byte)."),
    ("The timing as shipped: MPC_BUDGET_MS = 5.0 with MPC_MIN_HORIZON = MPC_HORIZON = 40, so the clock only counts; "
     "every race in this notebook ran it off except this check."),
    (f"check_submission on monza-like: shape and race passed here, ok {report.ok} with this machine's p99 "
     f"{report.step_p99_ms:.3f} ms, now; the "
     "timed check that counts is run on the race machine, idle, before the race."),
    ("The line (285 values) and the grip (MU_G) are monza-like's: on test-circuit the agent refuses to build "
     "(Unit 13, Exercise 8), so it is checked on its own track."),
    ("Versions written down (Python, numpy, racekit) and the file's SHA-256 recorded in roar/docs/qualification.md, "
     "so the file that raced can be proved."),
    ("The season's files (competition_runner.py, infrastructure.py) unmodified, and the season's page read fresh "
     "(instruction.txt or requirements.txt) with the date."),
    ("Racekit is a model: the final qualification (five official-runner launches on CARLA, all clean, median at most "
     "329 s) is the author's, and is not yet run."),
]
for number, item in enumerate(checklist, start=1):
    print(f"{number:2d}. {item}")

assert beats_bar and abs(bar_margin - 735.00) <= 1e-6 and bar_margin >= 700.0  # C14.1
assert beats_unit07 and unit07_clean == 5 and abs(np.median(unit07_times) - 515.10) <= 1e-6
assert check_passed and len(checklist) >= 8

## Milestone 5

### The engineering report

**Memo: the race-day file, `agents/unit-13.py`, and what each stage of it bought.**
Every number below is racekit's, on `monza-like`, 3 laps, `carla_like(noise=True)`, the timing off, race seeds 0 to 4
(from the tables above: the live seed-0 races and the probe record of plan `roar-014`).

**1. The table.**

| stage | idea | clean | median (s) | spread (s) | time bought (s) | better (Unit 1) | max error (m) |
|---|---|---|---|---|---|---|---|
| S0 | the bar: `starter.py`, `TARGET_SPEED` 14 | 5/5 | 1203.55 | 0.70 | | | 0.706 (c) |
| S1 | Unit 4: PID steering, flat 14 m/s | 5/5 | 1219.75 | 0.60 | −16.20 | False | 0.434 (c) |
| S2a | Unit 5: pure pursuit, the same 14 m/s | 5/5 | 1219.90 | 0.55 | −0.15 | False | 0.429 (c) |
| S2b | Unit 5's speed: pursuit at 15 m/s | 5/5 | 1138.85 | 0.45 | 81.05 | True | 1.042 (c) |
| S3 | Unit 6: the corner cap, 32.5 m/s | 5/5 | 617.05 | 0.15 | 521.80 | True | 0.753 (c) |
| S4 | Unit 7: the speed profile, 45 m/s | 5/5 | 514.95 | 0.20 | 102.10 | True | 1.575 (c) |
| S5 | Unit 8: the measured grip | 5/5 | 510.20 | 0.15 | 4.75 | True | 1.734 (c) |
| S6 | Unit 9: the racing line | 5/5 | 482.00 | 0.10 | 28.20 | True | 1.529 |
| S7 | Unit 10: MPC steering | 5/5 | 482.00 | 0.05 | 0.00 | False | 0.976 |
| S8 | Unit 11: the respawn fallback | 5/5 | 482.00 | 0.05 | 0.00 | False | 0.976 |
| **S9** | **Unit 13's re-tune: `agents/unit-13.py`, SUBMITTED** | 5/5 | 468.55 | 0.10 | 13.45 | True | 1.009 |
| S10 | the candidate, `TARGET_SPEED` 50.0: **not submitted** (test 23/24) | 5/5 | 447.90 | 0.20 | 20.65 | True | 1.141 |

(c): the error from the centreline (no line raced).
All 60 races finished with 0 collisions; seed 0 of every row was raced live here and matched the record to one tick.

**2. What each stage bought.**
The submitted file beats the bar by 735.00 s, and 688.60 s of it (94%) was bought before Unit 8, from the bar to the
speed profile (S4); Unit 6's corner cap alone bought 521.80 s, because it is the first stage that lets the car go
faster than 15 m/s anywhere.
After that the gains shrink: the profile 102.10 s, the line 28.20 s, the re-tune 13.45 s, the grip 4.75 s.
Every one of those is far outside its noise (the largest spread of any stage after S2b is 0.20 s), and Unit 1's rule
says so.
Within the noise: S2a (−0.15 s against spreads of 0.55 and 0.60), MPC and the fallback (0.00 s).

**3. The stage that cost time.**
S1, Unit 4's PID steering at the bar's 14 m/s, is 16.20 s slower than the bar it builds on.
The live seed-0 races show why: both cars drive 17.14 km in the three laps, but the starter's proportional speed loop
(gain 0.05) rides above its own target (a mean of 14.24 m/s), while Unit 4's PI loop holds 14.05 m/s; the bar is
not exactly a 14 m/s car.
What S1 bought instead is precision: its largest error from the centreline is 0.434 m against the bar's 0.706 m.
S2a against S2b says where a stage's value comes from: pursuit at the same 14 m/s buys nothing (−0.15 s, not better),
but it lets the car run 15 m/s clean, where PID at 15 m/s crashed on every seed, and that next clean m/s is worth
81.05 s.
A stage is worth what it lets you change next.

**4. What race time does not show.**
MPC (S7) bought 0.00 s, and the respawn fallback (S8) bought 0.00 s on a race that drove the same telemetry, row for
row, as S7: on the nominal car nothing ever respawns, so the fallback never acts.
What MPC bought is margin: the largest line error fell from 1.529 m to 0.976 m, half a metre further from the 2 m
clean limit, and Unit 13's record on its 24 held-out cars (24 to 47) has the race-day file clean on 24 of 24 with
MPC and on only 13 of 24 with pursuit.
The fallback's value is what a crash costs when one happens (Unit 11), on a car the nominal seeds never are.
A time-only ablation would call both useless; they are the reason the file survives cars it was not tuned on.

**5. The order, and why stages do not add.**
Removing one idea from the submitted file costs: the line 26.85 s (it bought 28.20 s in the build-up), the grip
17.70 s (it bought 4.75 s), MPC 0.05 s (not better by Unit 1's rule; it bought 0.00 s) and the re-tune 13.45 s (it
bought 13.45 s).
The grip is the clearest case: in the build-up it was added at `MU_FRACTION` 0.54, barely above the cap's `A_LAT` 4.6,
but the re-tune then raised the fraction to 0.64, so removing the grip from the finished file also removes most of
what the re-tune could use.
An idea's value depends on the ideas already there, so the time each stage bought belongs to this order (bar,
steering, speed, cap, profile, grip, line, MPC, fallback, re-tune), and a report must state it; the leave-one-out is
the cross-check that shows how much the order matters.

**6. The choice, and the file submitted.**
The knob was `TARGET_SPEED` (Unit 4's top speed, set on the kinematic car and never re-tuned after the grip, the line
and MPC raised the corner speeds), grid step 2.5 m/s, by Unit 13's protocol on whole ticks: 47.5, 50.0 and 52.5 each
beat the value before; 55.0 was not clean (car 8: one collision, 5.273 m); the edge is 52.5 and the margin value 50.0.
On the validation cars 24 to 47, 50.0 was clean on 24 of 24 (median 448.12 s) and 52.5, without the margin, on 21 of
24 (cars 24, 43 and 46 collide).
The second knob's evidence stops at validation: `PROFILE_LEAD_S` 0.2, 0.1 and 0.0 each beat the value before on
training, all clean, so with no edge 0.0 was kept, and it was clean on only 22 of the 24 validation cars (car 24
collided twice; car 35 ended 2.009 m off its line): no second candidate. `MPC_Q_E` 0.2 was slower, 0.5 not clean.
The candidate was pre-registered (dc170c9) on cars 96 to 119, raced once: **23 of 24 clean**; car 117 (race seed 2)
collided twice, 590.05 s, 4.485 m from its line.
**The submitted file is `agents/unit-13.py`**, clean on all 24 of its own test cars (72 to 95): Unit 13's rule, a file
clean on every car it was never tuned on, vetoes the candidate.
That submission is a later safety decision: the pre-registration named the cars and the file but no pass/fail rule,
and it should have.
The two facts stay apart: the candidate is 20.65 s faster on the nominal car (447.90 s against 468.55 s), and it
crashed on one car in 24.
They cannot be netted into one expected time: the two files' tests ran on different cars, one crash in 24 says the
crash rate is somewhere between a fraction of a percent and about 20%, and race day is one car, not an average.

**7. What next.**
The next choice would be `TARGET_SPEED` 47.5, one step further inside the edge, after looking for car 117's kind of
failure where cars may still inform (low-grip training and validation cars at race seed 2), never on car 117 again.
It would need a fresh set of 24 cars that nothing has raced, named before any race of them (not 72 to 143, not
k ≥ 1000), with the rule written down first: clean on all 24, and better than `agents/unit-13.py` by Unit 1's rule
on the nominal seeds 0 to 4.
Only that result would make me submit it.

**8. The CARLA caveat.**
Every number above is racekit's: `carla_like(noise=True)` is a model of CARLA's car on a model of Monza, so the
735 s margin, the 468.55 s and the 23 of 24 are claims about the model, and the ranking of two files is the most of
them likely to carry over.
On CARLA the submitted file is the lab-assembled one, with the author's own Monza line and grip.
The target that only CARLA can test is the book's final qualification: five official-runner launches on CARLA's
Monza, every one clean, median at most 329 s; it is not yet run, and the timed CARLA check of `agents/unit-13.py`
is still pending with the author.

In [ ]:
# The report's numbers, checked against the tables above.
assert abs(row_of["S0"]["median"] - row_of["S4"]["median"] - 688.60) <= 1e-6 and abs(bar_margin - 735.00) <= 1e-6
assert abs(row_of["S10"]["median"] - 447.90) <= 1e-6 and abs(row_of["S9"]["median"] - 468.55) <= 1e-6
assert abs(np.median(VALIDATION_TIMES[50.0]) - 448.125) <= 1e-6 and TEST_RECORD[117] == (590.05, 2, 4.485)
assert max(row["spread"] for row in ablation[4:]) <= 0.20 + 1e-9
print("the report's numbers match the tables")